# SATB Arranger Tool - Google Colab Version

This notebook allows you to run the SATB Arranger tool on Google Colab. It creates four-part SATB (Soprano, Alto, Tenor, Bass) arrangements from YouTube videos.

## Features:
- Download audio from YouTube videos
- Extract and separate vocal stems using Spleeter
- Analyze harmonic content and create four-part arrangements
- Generate individual MIDI files for each voice part (S, A, T, B)
- Audio playback and visualization

## Step 1: Install System Dependencies

First, we need to install FFmpeg and other system dependencies required for audio processing.

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y ffmpeg

## Step 2: Install Python Dependencies

Install all required Python packages for the SATB Arranger.

In [ ]:
# Install Python dependencies
!pip install -q spleeter==2.3.2
!pip install -q yt-dlp==2023.12.30
!pip install -q pydub==0.25.1
!pip install -q numpy==1.24.3
!pip install -q scipy==1.10.1
!pip install -q librosa==0.10.1
!pip install -q midiutil==1.2.1
!pip install -q music21==9.1.0
!pip install -q tensorflow==2.13.0
!pip install -q ffmpeg-python==0.2.0
!pip install -q streamlit==1.29.0
!pip install -q matplotlib==3.7.2
!pip install -q soundfile==0.12.1

print("All dependencies installed successfully!")

## Step 3: Clone the Repository or Upload Files

Choose one of the following options:
- **Option A**: Clone from a Git repository (if your project is on GitHub)
- **Option B**: Upload your project files manually

In [ ]:
# Option A: Clone from repository (uncomment and modify if using Git)
# !git clone https://github.com/yourusername/satb-arranger.git
# %cd satb-arranger

# Option B: Create project structure and upload files
import os

# Create directory structure
os.makedirs('satb-arranger/src', exist_ok=True)
os.makedirs('satb-arranger/output/audio', exist_ok=True)
os.makedirs('satb-arranger/output/stems', exist_ok=True)
os.makedirs('satb-arranger/output/midi', exist_ok=True)
os.makedirs('satb-arranger/models', exist_ok=True)

%cd satb-arranger

print("Project structure created. Please upload your source files to the 'src' folder.")

## Optional: Mount Google Drive

If you want to save outputs to Google Drive or load files from there.

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

## Step 4: Upload Source Files

**IMPORTANT**: You need to upload your source files to the `satb-arranger/src/` directory.

### Method 1: Using Colab File Browser
1. Click on the folder icon in the left sidebar
2. Navigate to `/content/satb-arranger/src/`
3. Right-click and select "Upload"
4. Upload these files:
   - `__init__.py`
   - `youtube_downloader.py`
   - `stem_separator.py`
   - `arranger.py`
   - `midi_generator.py`
   - `app.py`
   - `main.py`

### Method 2: Using Code Upload

In [ ]:
# Upload files programmatically
from google.colab import files
import shutil

print("Click 'Choose Files' to upload your source files")
uploaded = files.upload()

# Move uploaded files to src directory
for filename in uploaded.keys():
    if filename.endswith('.py'):
        shutil.move(filename, f'src/{filename}')
        print(f"Moved {filename} to src/")

## Step 5: Test Import

Let's verify that all modules can be imported correctly.

In [ ]:
# Add src to Python path
import sys
sys.path.append('src')

# Test imports
try:
    from youtube_downloader import YouTubeDownloader
    from stem_separator import StemSeparator
    from arranger import SATBArranger
    from midi_generator import MIDIGenerator
    print("✅ All modules imported successfully!")
except Exception as e:
    print(f"❌ Import error: {e}")
    print("Please make sure you've uploaded all source files to the src/ directory.")

## Step 6: Download Spleeter Models

Spleeter needs to download its pre-trained models on first use.

In [ ]:
# Download Spleeter models
from spleeter import separator

# This will download the 2-stem model (vocals/accompaniment)
sep = separator.Separator('spleeter:2stems')
print("✅ Spleeter 2-stem model downloaded")

# Optionally download other models
# sep4 = separator.Separator('spleeter:4stems')  # vocals/drums/bass/other
# sep5 = separator.Separator('spleeter:5stems')  # vocals/drums/bass/piano/other

## Step 7: Create SATB Arrangement Function

Here's the main function to create SATB arrangements from YouTube URLs.

In [ ]:
def create_satb_arrangement(youtube_url, style='hymn'):
    """Create SATB arrangement from YouTube URL"""
    
    # Initialize components
    downloader = YouTubeDownloader(output_dir="output/audio")
    separator = StemSeparator(output_dir="output/stems")
    arranger = SATBArranger()
    midi_gen = MIDIGenerator(output_dir="output/midi")
    
    try:
        # Step 1: Download audio
        print("📥 Downloading audio from YouTube...")
        video_info = downloader.get_video_info(youtube_url)
        if video_info:
            print(f"Video: {video_info['title']} by {video_info['uploader']}")
            
        audio_path = downloader.download_audio(youtube_url)
        print(f"✅ Audio saved to: {audio_path}")
        
        # Step 2: Separate stems
        print("\n🎵 Separating vocals from accompaniment...")
        stems = separator.separate_vocals(audio_path, stems=2)
        vocal_path = stems['vocals']
        print(f"✅ Vocals extracted to: {vocal_path}")
        
        # Step 3: Analyze melody
        print("\n🎼 Analyzing vocal melody...")
        melody_data = separator.extract_vocal_melody(vocal_path)
        print(f"✅ Tempo: {melody_data['tempo']:.0f} BPM")
        print(f"✅ Key: {melody_data['key']}")
        
        # Step 4: Create arrangement
        print(f"\n🎹 Creating SATB arrangement ({style} style)...")
        parts = arranger.create_arrangement(melody_data, arrangement_style=style)
        print("✅ Four-part harmony created")
        
        # Step 5: Export MIDI
        print("\n💾 Exporting MIDI files...")
        midi_gen.set_midi_instruments(parts)
        midi_paths = midi_gen.export_parts(parts)
        combined_path = midi_gen.export_combined(parts)
        
        print("\n✅ Success! Files created:")
        for voice, path in midi_paths.items():
            print(f"   - {voice}: {path}")
        print(f"   - Combined: {combined_path}")
        
        return midi_paths, combined_path
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        raise

## Step 8: Interactive Interface

Use this interactive interface to easily create arrangements.

In [ ]:
# Simple input form
youtube_url = input("Enter YouTube URL: ")
style = input("Enter style (hymn/jazz/classical) [default: hymn]: ") or "hymn"

if youtube_url:
    midi_paths, combined_path = create_satb_arrangement(youtube_url, style)
else:
    print("Please enter a valid YouTube URL")

## Alternative: Command Line Usage

You can also use the command line interface directly.

In [ ]:
# Example command line usage
# Replace YOUR_VIDEO_ID with actual YouTube video ID
# !python src/main.py --youtube-url "https://www.youtube.com/watch?v=YOUR_VIDEO_ID" --output-dir ./output --style hymn

## Utility Functions

Helper functions for managing output files.

In [ ]:
import os
import zipfile
from IPython.display import Audio, display

def list_output_files():
    """List all generated output files"""
    output_dirs = ['output/audio', 'output/stems', 'output/midi']
    
    for dir_path in output_dirs:
        if os.path.exists(dir_path):
            print(f"\n📁 {dir_path}:")
            for file in os.listdir(dir_path):
                file_path = os.path.join(dir_path, file)
                if os.path.isfile(file_path):
                    size = os.path.getsize(file_path) / 1024  # KB
                    print(f"   - {file} ({size:.1f} KB)")

def create_download_zip():
    """Create a zip file of all outputs for download"""
    zip_path = 'satb_arrangement_output.zip'
    
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for root, dirs, files in os.walk('output'):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, 'output')
                zipf.write(file_path, arcname)
    
    print(f"✅ Created {zip_path} for download")
    return zip_path

def play_audio(audio_path):
    """Play audio file in Colab"""
    display(Audio(audio_path))

# List files
list_output_files()

## Download Your Results

Download all generated files as a zip archive.

In [ ]:
from google.colab import files

# Create and download zip file
zip_path = create_download_zip()
files.download(zip_path)
print("\n✅ Download started!")

## Tips and Troubleshooting

### Common Issues:

1. **Import errors**: Make sure all source files are uploaded to the `src/` directory
2. **FFmpeg not found**: Re-run the FFmpeg installation cell
3. **Spleeter model download fails**: Check your internet connection
4. **Out of memory**: Use Runtime > Change runtime type to get more RAM
5. **YouTube download fails**: Video might be restricted or URL invalid

### Performance Tips:
- Use shorter videos (< 5 minutes) for faster processing
- The 'hymn' style works best for traditional choral music
- Enable GPU in Runtime settings for faster processing

### Save to Google Drive:
```python
# Save outputs to Google Drive
!cp -r output /content/drive/MyDrive/SATB_Outputs/
```